In [1]:
# Dependency install cell (ultimate one-shot setup for Kani + Qwen HF on SageMaker)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists (SageMaker usually already has it)
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

> /opt/conda/bin/python -m pip uninstall -y torchvision transformers tokenizers kani


Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: tokenizers 0.21.1
Uninstalling tokenizers-0.21.1:
  Successfully uninstalled tokenizers-0.21.1
Found existing installation: kani 1.8.0
Uninstalling kani-1.8.0:
  Successfully uninstalled kani-1.8.0
> /opt/conda/bin/python -m pip install --no-cache-dir -U kani[huggingface]==1.8.0 transformers==4.51.3 tokenizers==0.21.1 accelerate>=0.30.0 huggingface_hub>=0.30.0 sentencepiece safetensors protobuf nest_asyncio python-dotenv tqdm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 351.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 517.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-multimodal 1.5.0 requires torchvision<0.25.0,>=0.21.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-multimodal 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.10.0 which is incompatible.
autogluon-timeseries 1.5.0 requires torch<2.10,>=2.6, but you have torch 2.10.0 which is incompatible.


torch already available

Install complete. IMPORTANT: Restart kernel now, then run Cell 1.


In [1]:
# Quick test cell (run after kernel restart)

import importlib.metadata as md

print("python:", __import__("sys").executable)
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

python: /opt/conda/bin/python
kani: 1.8.0
torch: 2.10.0
transformers: 4.51.3
accelerate: 1.13.0
tokenizers: 0.21.1
huggingface_hub: 0.36.2


2026-03-09 07:29:34.842111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773041374.858295   13446 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773041374.863824   13446 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-09 07:29:34.881196: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Kani import OK
HuggingEngine import OK


In [2]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (DNC, independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Put heavy caches on NVMe (recommended in SageMaker)
NVME_CACHE_ROOT = Path("/mnt/sagemaker-nvme/formalizer_cache")
HF_HOME_DIR = NVME_CACHE_ROOT / "hf_home"
HF_HUB_CACHE_DIR = NVME_CACHE_ROOT / "hf_hub"
TRANSFORMERS_CACHE_DIR = NVME_CACHE_ROOT / "transformers"
TORCH_HOME_DIR = NVME_CACHE_ROOT / "torch"
XDG_CACHE_DIR = NVME_CACHE_ROOT / "xdg_cache"

for d in [
  NVME_CACHE_ROOT,
  HF_HOME_DIR,
  HF_HUB_CACHE_DIR,
  TRANSFORMERS_CACHE_DIR,
  TORCH_HOME_DIR,
  XDG_CACHE_DIR,
]:
  d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE_DIR)
os.environ["TORCH_HOME"] = str(TORCH_HOME_DIR)
os.environ["XDG_CACHE_HOME"] = str(XDG_CACHE_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DATA_DIR = PROJECT_ROOT / "data"
DESC_DIR = DATA_DIR / "descriptions"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"

PROMPT_HEADER_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction_dnc_header.txt"
PROMPT_LINE_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction_dnc_line.txt"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_dnc" / "qwen2_5_32b_instruct_kani"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Model setup (same model for LLM-1 header and LLM-2 line generation)
MODEL_NAME = "Qwen2.5-32B-Instruct"
HF_MODEL_ID = f"Qwen/{MODEL_NAME}"

QWEN_ENGINE = HuggingEngine(
  model_id=HF_MODEL_ID,
  model_load_kwargs={
      "device_map": "auto",
      "cache_dir": str(TRANSFORMERS_CACHE_DIR),
  },
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Header prompt exists:", PROMPT_HEADER_PATH.exists())
print("Line prompt exists:", PROMPT_LINE_PATH.exists())
print("Model:", HF_MODEL_ID)
print("NVME cache root:", NVME_CACHE_ROOT)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
No handwritten prompt pipeline was found for the Qwen/Qwen2.5-32B-Instruct model. Kani is falling back to the Hugging Face chat template. For most models this is okay, but you may want to verify that the chat template correctly passes tool calls to the LLM.


PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_dnc/qwen2_5_32b_instruct_kani
VAL exists: True
Domain PDDL exists: True
Header prompt exists: True
Line prompt exists: True
Model: Qwen/Qwen2.5-32B-Instruct
NVME cache root: /mnt/sagemaker-nvme/formalizer_cache


In [3]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]  # p01..p200

PROMPT_HEADER_PREFIX = PROMPT_HEADER_PATH.read_text()
PROMPT_LINE_PREFIX = PROMPT_LINE_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Header prompt chars:", len(PROMPT_HEADER_PREFIX))
print("Line prompt chars:", len(PROMPT_LINE_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 200
First/last: p01 p200
Header prompt chars: 0
Line prompt chars: 0
Domain PDDL chars: 997


In [4]:
# Cell 3: Load NL descriptions for one problem ID (original + line-parsed)
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"
  problem_line_path = DESC_DIR / f"{problem_id}_problem_line.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_path}")
  if not problem_line_path.exists():
      raise FileNotFoundError(f"Missing line-parsed problem NL file: {problem_line_path}")

  return {
      "domain_nl": domain_path.read_text(),
      "problem_nl": problem_path.read_text(),
      "problem_nl_line": problem_line_path.read_text(),
  }

In [ ]:
# Cell 4: Build prompts for one problem (DNC header stage + DNC line stage)
def build_header_prompt(problem_id: str):
  parts = load_problem_descriptions(problem_id)
  nl_domain = parts["domain_nl"]
  nl_problem = parts["problem_nl"]

  return (
      f"{PROMPT_HEADER_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Generate ONLY the problem header/object parts "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the problem header/object parts of the problem PDDL for this problem.\n"
      "- Wrap it in exactly one <header>...</header> block.\n"
      "- Do not generate init or goal facts.\n"
      "- Follow the exact formatting constraints in the instruction.\n"
      "- Do not include any extra text before or after <header>...</header>."
  )


def build_line_prompt(problem_id: str, section: str, line_text: str):
  parts = load_problem_descriptions(problem_id)
  nl_domain = parts["domain_nl"]
  nl_problem_line = parts["problem_nl_line"]

  return (
      f"{PROMPT_LINE_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Read one line and generate ONLY its corresponding PDDL fact line.\n\n"
      "You are given four inputs:\n"
      "1) Original natural language domain description\n"
      "2) Line-parsed natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n"
      "4) Current single natural language line to convert\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Line-parsed problem description\n"
      f"{nl_problem_line}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "### Section\n"
      f"{section}\n\n"
      "### Current line\n"
      f"{line_text}\n\n"
      "Output contract:\n"
      "- Generate ONLY one corresponding PDDL fact line.\n"
      "- Do not include extra explanation or text."
  )